# LoRA and QLoRA from First Principles

## A mathematical, computational, and research-level journey

This notebook starts before LoRA. It rebuilds every prerequisite: vectors, matrices, linear maps, matrix rank, low-rank factorization, gradients, and full fine-tuning. It then derives LoRA, implements it from scratch in PyTorch, compares it with full fine-tuning, places it inside transformer attention, develops quantization and QLoRA, and closes with research questions and a machine-unlearning experiment design.

Learning goals:

1. Explain every symbol in the LoRA equation.
2. Calculate the parameter and memory savings.
3. Understand what rank means geometrically and algebraically.
4. Implement a LoRA layer without a library.
5. Explain exactly what still undergoes backpropagation.
6. Distinguish LoRA, QLoRA, RAG, prompting, and full fine-tuning.
7. Connect parameter-efficient adaptation to the harder problem of machine unlearning.

Run the cells in order. GPU is optional except for the final optional Hugging Face lab.

In [ ]:
import math, copy, random, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 1. What is being adapted?

A neural network is a parameterized function:

$$f_\theta(x)=y$$

- $x$ is the input.
- $y$ is the prediction.
- $\theta$ denotes all trainable parameters.

A transformer's parameters are dominated by matrices. A linear layer computes:

$$y=Wx+b$$

where $x\in\mathbb{R}^{d_{in}}$, $W\in\mathbb{R}^{d_{out}\times d_{in}}$, $b\in\mathbb{R}^{d_{out}}$, and $y\in\mathbb{R}^{d_{out}}$.

A matrix is not merely a table. It is a linear transformation: it can rotate, stretch, compress, reflect, or project vectors. In a transformer, matrices create queries, keys, values, attention outputs, and feed-forward transformations. Adapting a model means changing some of these transformations.

Shape rule: the inner dimensions must agree. A $3\times4$ matrix consumes a four-dimensional vector and produces a three-dimensional vector.

Pause and predict:

1. If $W$ is $4096\times4096$, what is the shape of $Wx$?
2. How many scalars are in $W$?
3. At two bytes per scalar, how many MiB does it occupy?

In [ ]:
W = torch.tensor([[1.,2.,0.,-1.],[.5,0.,1.,1.],[2.,-1.,.5,0.]])
x = torch.tensor([1.,2.,-1.,.5])
b = torch.tensor([.1,-.2,.3])
print("W shape:", W.shape, "x shape:", x.shape)
print("W @ x + b =", W @ x + b)
n = 4096 * 4096
print(f"4096-square parameters: {n:,}")
print(f"FP16 storage: {n*2/1024**2:.1f} MiB")

# 2. Rank from the beginning

The rank of a matrix is the number of linearly independent directions represented by its rows, equivalently its columns.

Two vectors are linearly dependent if one can be constructed from the others. In

$$M=\begin{bmatrix}1&2\\2&4\end{bmatrix},$$

the second row is twice the first. The matrix has rank 1, not 2.

Rank answers: how many independent dimensions of transformation does this matrix actually contain?

## Outer products

Let $b\in\mathbb{R}^{m}$ and $a\in\mathbb{R}^{n}$. The outer product $ba^T$ is an $m\times n$ matrix of rank at most 1. Every column is a scaled copy of $b$.

A rank-$r$ matrix can be represented as a sum of $r$ rank-1 outer products:

$$\Delta W=\sum_{i=1}^{r}b_i a_i^T.$$

Stack the $b_i$ vectors as columns of $B$ and the $a_i^T$ vectors as rows of $A$:

$$\Delta W=BA,$$

where

$$B\in\mathbb{R}^{d_{out}\times r},\qquad A\in\mathbb{R}^{r\times d_{in}}.$$

This is the mathematical doorway into LoRA. Because the product passes through an $r$-dimensional middle space, $\operatorname{rank}(BA)\le r$.

In [ ]:
M1 = torch.tensor([[1.,2.],[2.,4.]])
M2 = torch.tensor([[1.,2.],[2.,5.]])
print("rank M1:", torch.linalg.matrix_rank(M1).item())
print("rank M2:", torch.linalg.matrix_rank(M2).item())

b_vec = torch.tensor([1.,2.,3.])
a_vec = torch.tensor([2.,-1.,.5,4.])
outer = b_vec[:,None] @ a_vec[None,:]
print("\nOuter product:\n", outer)
print("shape:", outer.shape, "rank:", torch.linalg.matrix_rank(outer).item())

## Singular Value Decomposition

Any matrix $M\in\mathbb{R}^{m\times n}$ can be decomposed:

$$M=U\Sigma V^T.$$

- $U$ contains orthonormal output directions.
- $V$ contains orthonormal input directions.
- $\Sigma$ contains nonnegative singular values.
- Large singular values correspond to influential directions.

The best rank-$r$ approximation under the Frobenius norm keeps the largest $r$ singular values:

$$M_r=U_{:,:r}\Sigma_{:r,:r}V_{:,:r}^T.$$

LoRA normally does not calculate an SVD of the desired update beforehand. It learns $A$ and $B$ directly. SVD gives us intuition for why a low-rank representation may work.

Rank is not parameter count. A dense $m\times n$ update stores $mn$ values. The factors store $mr+rn=r(m+n)$ values. For $m=n=4096$ and $r=8$, that is 16,777,216 versus 65,536 parameters: 256 times fewer.

In [ ]:
torch.manual_seed(7)
M = torch.randn(30,3) @ torch.randn(3,20) + .05*torch.randn(30,20)
U,S,Vh = torch.linalg.svd(M, full_matrices=False)
print("Singular values:", S[:10])

errors=[]
for r in range(1,16):
    Mr=U[:,:r] @ torch.diag(S[:r]) @ Vh[:r,:]
    errors.append((torch.linalg.norm(M-Mr)/torch.linalg.norm(M)).item())
plt.plot(range(1,16),errors,marker="o")
plt.xlabel("Approximation rank r"); plt.ylabel("Relative error")
plt.grid(); plt.show()

def report(din,dout,r):
    dense=din*dout; low=r*(din+dout)
    return dense,low,dense/low
for r in [1,2,4,8,16,64,256]:
    dense,low,factor=report(4096,4096,r)
    print(f"r={r:3}: {low:,} vs {dense:,}; reduction={factor:.1f}x")

# 3. Full fine-tuning and backpropagation

Let $W_0$ be a pretrained matrix. Full fine-tuning searches for:

$$W^*=W_0+\Delta W.$$

Gradient descent updates every element:

$$W_{t+1}=W_t-\eta\frac{\partial\mathcal L}{\partial W_t},$$

where $t$ is the step, $\eta$ is the learning rate, $\mathcal L$ is the loss, and the derivative is a matrix of gradients.

A crucial correction: LoRA does not avoid backpropagation. Gradients still flow through the forward computation. LoRA avoids training, storing gradients for, and maintaining optimizer states for the frozen base weights. Only the small adapter factors update.

Training memory includes parameters, gradients, optimizer states, saved activations, and temporary buffers. Adam commonly stores first and second moment estimates for every trainable parameter. Freezing billions of parameters therefore saves much more than merely the adapter file size.

# 4. Deriving LoRA

Full fine-tuning permits an unrestricted update:

$$h=W_0x+\Delta Wx.$$

LoRA constrains it:

$$\Delta W=BA.$$

The resulting computation is:

$$\boxed{h=W_0x+\frac{\alpha}{r}BAx}.$$

Every symbol:

- $h\in\mathbb{R}^{d_{out}}$: output activation.
- $x\in\mathbb{R}^{d_{in}}$: input activation.
- $W_0\in\mathbb{R}^{d_{out}\times d_{in}}$: frozen pretrained matrix.
- $A\in\mathbb{R}^{r\times d_{in}}$: trainable down-projection.
- $B\in\mathbb{R}^{d_{out}\times r}$: trainable up-projection.
- $r$: adapter rank.
- $\alpha$: scaling hyperparameter.
- $\alpha/r$: conventional scale.

Read it right to left:

1. $Ax$ compresses $d_{in}$ features into $r$ adapter coordinates.
2. $B(Ax)$ expands them into $d_{out}$ output coordinates.
3. The scaled correction is added to the frozen output.

A common initialization makes $A$ random and $B=0$. Initially $BA=0$, so the adapted model exactly matches the pretrained model. After the first backward pass, $B$ receives a gradient. Once $B$ is nonzero, $A$ begins receiving useful gradients. If both factors start at zero, neither gets the useful symmetry-breaking signal needed to start learning.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8., dropout=0.):
        super().__init__()
        if rank <= 0: raise ValueError("rank must be positive")
        self.rank, self.alpha = rank, alpha
        self.scaling = alpha/rank
        self.dropout = nn.Dropout(dropout)
        self.base = copy.deepcopy(base)
        for p in self.base.parameters(): p.requires_grad=False
        self.A = nn.Parameter(torch.empty(rank,base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        base_y=self.base(x)
        # F.linear(x,A) is x @ A.T
        adapter_y=F.linear(F.linear(self.dropout(x),self.A),self.B)
        return base_y+self.scaling*adapter_y

    def merged_weight(self):
        return self.base.weight+self.scaling*(self.B@self.A)

torch.manual_seed(1)
base=nn.Linear(5,3)
lora=LoRALinear(base,rank=2,alpha=4)
sample=torch.randn(7,5)
print("Initial max difference:",(base(sample)-lora(sample)).abs().max().item())
for name,p in lora.named_parameters():
    print(name,tuple(p.shape),"trainable=",p.requires_grad)

with torch.no_grad(): lora.B.normal_(0,.1)
unmerged=lora(sample)
merged=F.linear(sample,lora.merged_weight(),lora.base.bias)
print("Merge discrepancy:",(unmerged-merged).abs().max().item())

# 5. Controlled adaptation experiment

We simulate pretraining and domain adaptation:

1. A base model learns task A.
2. Target task B is related but shifted.
3. The frozen model exposes the domain gap.
4. Full fine-tuning updates every parameter.
5. LoRA freezes the base and learns low-rank adapters.

This is not an LLM. Its scientific advantage is causal clarity: every parameter and data-generating rule is inspectable.

In [ ]:
def make_data(n,shift=False):
    x=torch.randn(n,12)
    w=torch.tensor([1.2,-.7,0,.4,-1,.3,0,.8,-.2,0,.5,-.4])
    if shift: w=w+torch.tensor([.3,0,0,-.2,0,0,.4,0,0,-.3,0,.2])
    y=(x@w+.15*torch.sin(x[:,0]*x[:,1])).unsqueeze(1)
    return x,y

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(12,32); self.fc2=nn.Linear(32,32); self.out=nn.Linear(32,1)
    def forward(self,x):
        return self.out(F.gelu(self.fc2(F.gelu(self.fc1(x)))))

def train(model,x,y,steps=400,lr=3e-3):
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr)
    losses=[]
    for _ in range(steps):
        opt.zero_grad(); loss=F.mse_loss(model(x),y); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

def mse(model,x,y):
    model.eval()
    with torch.no_grad(): return F.mse_loss(model(x),y).item()

def counts(model):
    return sum(p.numel() for p in model.parameters()),sum(p.numel() for p in model.parameters() if p.requires_grad)

xa,ya=make_data(1500); xb,yb=make_data(500,True); xt,yt=make_data(1000,True)
base_model=TinyModel(); train(base_model,xa,ya,700)

full_model=copy.deepcopy(base_model)
full_losses=train(full_model,xb,yb,400,2e-3)

lora_model=copy.deepcopy(base_model)
lora_model.fc1=LoRALinear(lora_model.fc1,4,8)
lora_model.fc2=LoRALinear(lora_model.fc2,4,8)
lora_model.out=LoRALinear(lora_model.out,2,4)
lora_losses=train(lora_model,xb,yb,400,5e-3)

print("Frozen target MSE:",mse(base_model,xt,yt))
print("Full target MSE:",mse(full_model,xt,yt),"params:",counts(full_model))
print("LoRA target MSE:",mse(lora_model,xt,yt),"params:",counts(lora_model))
plt.plot(full_losses,label="full"); plt.plot(lora_losses,label="LoRA")
plt.yscale("log"); plt.legend(); plt.grid(); plt.show()

## How to interpret the result

Do not conclude that LoRA always matches full fine-tuning. Outcomes depend on:

- distance between target task and pretraining;
- adapter rank and placement;
- data quantity and quality;
- learning rate, alpha, dropout, and duration;
- whether the useful update is approximately low-rank.

Full fine-tuning has a larger hypothesis space. LoRA imposes a constraint. The constraint reduces cost and can regularize learning, but it can underfit.

Research exercise: repeat with ranks 1, 2, 4, 8, and 16 under identical seeds. Plot test error against trainable parameter count.

# 6. LoRA inside transformer attention

For token representations $X\in\mathbb{R}^{T\times d_{model}}$:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

Attention is:

$$\operatorname{Attention}(Q,K,V)=
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V.$$

$T$ is sequence length, $d_k$ is key dimension, and $M$ is an optional causal/padding mask.

A query adapter gives:

$$Q=X\left(W_Q+\frac{\alpha}{r}B_QA_Q\right).$$

LoRA may target:

- query and value projections only;
- query, key, value, and output projections;
- MLP projections;
- sometimes embeddings or the language-model head.

More target modules add capacity and memory. There is no universally optimal selection.

Important hyperparameters:

- Rank $r$: maximum adapter-update rank.
- Alpha $\alpha$: controls update scale; it is not a second rank.
- Dropout: regularizes the adapter path.
- Learning rate: adapters often tolerate higher rates than full fine-tuning.
- Target modules: determines where representational behavior may change.
- Layer coverage: adapting all layers is different from adapting only later layers.

Adapters are swappable and can be merged into the base weights for inference. Combining multiple adapters is not automatically safe because their updates may interfere.

# 7. Quantization and QLoRA

Quantization represents numbers using fewer bits. A simple uniform quantizer maps:

$$q=\operatorname{round}\left(\frac{w}{s}\right)+z,$$

and approximately reconstructs:

$$\hat w=s(q-z).$$

Here $s$ is scale, $z$ is zero-point, $q$ is the stored code, and $\hat w$ is the approximation.

Four bits provide only 16 codes per quantization group. Real LLM quantizers use group-wise scales and specialized formats rather than one global scale.

QLoRA combines:

1. a frozen base stored in a low-bit representation, commonly 4-bit;
2. trainable LoRA adapters at higher precision;
3. higher-precision computation where necessary.

Conceptually:

$$h=\operatorname{Dequantize}(W_q)x+\frac{\alpha}{r}BAx.$$

$W_q$ remains frozen. Only $A$ and $B$ receive optimizer updates.

The original QLoRA method also uses important engineering ideas such as a 4-bit format designed for normally distributed weights, double quantization, and paged optimizers. QLoRA therefore means more than naively rounding each weight.

Quantization changes the effective frozen computation. LoRA then learns on top of that approximation. Memory falls dramatically, but compute, data quality, and evaluation remain essential.

In [ ]:
def quant_dequant(x,bits=4):
    qmax=2**(bits-1)-1
    scale=x.abs().max()/qmax
    q=torch.clamp(torch.round(x/scale),-qmax,qmax)
    return q,q*scale,scale

values=torch.linspace(-1,1,17)**3
q,reconstructed,scale=quant_dequant(values)
print(torch.stack([values,q,reconstructed],dim=1))
print("Mean absolute error:",(values-reconstructed).abs().mean().item())

# 8. Optional Colab QLoRA skeleton

Run these in a fresh GPU runtime. Package versions and model access evolve, so check current model cards and documentation.

Installation:

    !pip -q install -U transformers datasets peft accelerate bitsandbytes

Core setup:

    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType
    import torch

    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=quant_config, device_map="auto"
    )

    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        bias="none",
    )

    model = get_peft_model(model, config)
    model.print_trainable_parameters()

Before training, answer:

1. Do these target-module names exist in this architecture?
2. Which parameters have requires_grad=True?
3. What fraction is trainable?
4. What is GPU memory usage?
5. What held-out evaluation distinguishes generalization from memorization?

# 9. Choosing the correct adaptation method

| Method | Changes weights? | Best suited to | Main limitation |
|---|---:|---|---|
| Prompting | No | Instructions and one-off context | Temporary and context-limited |
| RAG | No | Current, attributable external knowledge | Retrieval can fail |
| LoRA | Adapter weights only | Behavior, style, domain/task adaptation | Constrained update |
| QLoRA | Adapter only; quantized base | Memory-efficient adaptation | Quantization complexity |
| Full fine-tuning | Most/all weights | Maximum adaptation capacity | Expensive and riskier |

Do not fine-tune merely to inject rapidly changing facts. RAG generally offers better freshness, attribution, and deletion. Fine-tuning is useful when the model must learn a behavior, representation, format, or domain transformation.

# 10. Connection to machine unlearning

Suppose a synthetic fact is introduced only through an adapter. Removing that adapter removes its learned update from deployed computation. This is modular rollback, not a general solution to machine unlearning.

Harder cases occur when:

- the fact already exists in the base;
- it is redundantly distributed;
- related facts reconstruct it;
- the model learned a general rule, not one association;
- the model learned refusal while retaining recoverable information.

Possible forgetting criteria:

1. no direct elicitation;
2. no paraphrase elicitation;
3. no adversarial recovery;
4. no detectable internal representation under specified probes;
5. minimal damage to unrelated capabilities;
6. statistical equivalence to a model never trained on the target data.

These are not equivalent. “The model does not say it” is weaker than “the model no longer contains usable information about it.”

Proposed experiment:

1. Choose a small open model.
2. Create fictional facts, such as “Velora's currency is the nirin.”
3. Measure baseline outputs.
4. Teach facts through a LoRA adapter.
5. Test direct questions, paraphrases, implications, and adversarial prompts.
6. Remove the adapter and repeat.
7. Train a second adapter intended to negate the fact.
8. Determine whether behavior reflects forgetting, substitution, uncertainty, or refusal.
9. Measure collateral damage with control facts and general benchmarks.

# 11. Research-level questions

1. Why should useful task updates be low-rank? Is low intrinsic dimension a property of the task, representation, optimization, or all three?
2. Where should adapters be placed: attention, MLPs, embeddings, or dynamically selected modules?
3. Should each layer have the same rank? Adaptive-rank methods allocate capacity differently.
4. What does an adapter encode: behavior, facts, style, or a mixture?
5. How do independently trained adapters interfere when composed?
6. Do singular directions of $BA$ correspond to interpretable concepts?
7. When does low rank fail under large domain shifts?
8. Does QLoRA optimize the same solution as LoRA? Quantization changes the base and loss landscape.
9. Under what assumptions can adapter deletion qualify as certified unlearning?

Exercises:

- Prove $\operatorname{rank}(BA)\le r$.
- Calculate parameters for $d_{in}=4096$, $d_{out}=11008$, and $r=8,16,64$.
- Explain why rank 1 is not the same as changing one parameter.
- Find the rank preserving 95% of squared singular-value energy.
- Verify frozen weights receive no gradients.
- Implement a merge method returning a standard Linear layer.
- Compare ranks and target-layer choices under controlled seeds.
- Repeat adaptation with only 50 examples and several dropout values.

In [ ]:
energy=S.square()
cumulative=energy.cumsum(0)/energy.sum()
r95=int(torch.where(cumulative>=.95)[0][0])+1
print("Rank retaining at least 95% spectral energy:",r95)

# Gradient check
test=LoRALinear(nn.Linear(5,3),rank=2,alpha=4)
loss=test(torch.randn(8,5)).square().mean()
loss.backward()
for name,p in test.named_parameters():
    print(name,"gradient is None:",p.grad is None)

# 12. Final conceptual map

- A pretrained model contains large learned matrices.
- Full fine-tuning learns unrestricted changes to those matrices.
- LoRA freezes them and learns low-rank changes $BA$.
- Rank $r$ is the maximum number of independent update directions, not an accuracy score.
- LoRA still uses forward propagation, a loss, and backpropagation.
- Efficiency comes from training and optimizing far fewer parameters.
- QLoRA additionally stores the frozen base in a low-bit representation.
- LoRA, RAG, prompting, and full fine-tuning solve different problems.
- Removing an adapter is modular rollback, not general unlearning of information distributed through the base.

The one equation to retain:

$$\boxed{h=W_0x+\frac{\alpha}{r}BAx}.$$

Read it as:

> Preserve the pretrained transformation, then learn a small, structured correction.

Suggested next notebook: implement attention from scratch, inject LoRA separately into $W_Q$, $W_K$, $W_V$, and $W_O$, then measure how each target changes attention patterns and task behavior.